<a href="https://colab.research.google.com/github/rikadamay/rikadamay/blob/main/Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
pip install openai==0.28

In [12]:
%%writefile requirements.txt

streamlit==1.52.0
pyngrok==7.5.0
openai
python-dotenv==1.2.1
langchain==0.2.16
langchain-core==0.2.41
langchain-community==0.2.11
langchain-text-splitters==0.2.4
langgraph==0.2.3
langchain-ollama==0.1.1
semantic-router==0.0.61
pyppeteer==2.0.0
nest-asyncio==1.6.0
praw==7.7.1
cohere==5.5.0
replicate==1.0.7

Overwriting requirements.txt


In [13]:
from google.colab import userdata
import os

ngrok_token = userdata.get('ngrok_token')
api_token = userdata.get('api_token')
openai_token = userdata.get('openai_token')

# Put token to env variable
os.environ["OPENAI_API_KEY"] = openai_token

with open(".env", "w") as f:
    f.write(f"OPENAI_API_KEY={openai_token}")

In [27]:
%%writefile yummy_bot_app.py

import os
import streamlit as st
import requests
import json
from typing import List, Dict

# -------------------------
# Config & Helpers
# -------------------------
DEFAULT_MODEL = "gpt-4o-mini"  # change to your preferred model
API_URL = "https://api.openai.com/v1/chat/completions"

st.set_page_config(page_title="YummyBot", layout="wide")

PROMPT_SYSTEM = (
    "You are YummyBot, an expert and friendly AI chef assistant.\n"
    "- Create helpful recipes based on available ingredients.\n"
    "- Offer clear, step-by-step cooking instructions.\n"
    "- Suggest substitutions and variations.\n"
    "- Give estimated time, difficulty, and serving sizes.\n"
    "- Provide storage tips and plating suggestions.\n"
    "Keep tone warm, encouraging, and practical."
)


def call_openai_chat(api_key: str, messages: List[Dict], model: str = DEFAULT_MODEL, temperature: float = 0.7):
    """
    Simple wrapper to call OpenAI Chat Completions via REST API.
    Returns the assistant text or raises an exception.
    """
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": 900,
    }
    resp = requests.post(API_URL, headers=headers, json=payload, timeout=30)
    if resp.status_code != 200:
        raise Exception(f"OpenAI API error {resp.status_code}: {resp.text}")
    data = resp.json()
    # response structure may vary by model — pick first choice
    return data["choices"][0]["message"]["content"]


# -------------------------
# UI: Sidebar
# -------------------------
st.sidebar.title("YummyBot - Settings")
mode = st.sidebar.radio("Mode", (
    "Masak dengan Bahan di Rumah",
    "Pengganti Bahan",
    "Analisis Resep",
    "Planner 7 Hari",
))

st.sidebar.markdown("---")
api_key_input = st.sidebar.text_input("OpenAI API Key", type="password")
if not api_key_input:
    # also accept environment variable
    api_key_input = os.getenv("OPENAI_API_KEY", "")

model = st.sidebar.text_input("Model", value=DEFAULT_MODEL)
if not model:
    model = DEFAULT_MODEL

temperature = st.sidebar.slider("Creativity (temperature)", 0.0, 1.0, 0.7, 0.05)


# -------------------------
# Session state defaults
# -------------------------
if "history" not in st.session_state:
    st.session_state.history = []  # list of dicts: {mode, input, output}

if "favorites" not in st.session_state:
    st.session_state.favorites = []


# -------------------------
# Page Layout
# -------------------------
st.title("🍳 YummyBot - Chatbot Chef Pintar")
st.write("Bilang bahan yang kamu punya, atau pilih mode di sidebar. YummyBot akan bantu racik resep, substitusi bahan, atau bikin meal plan.")

col1, col2 = st.columns([2, 1])

with col1:
    if mode == "Masak dengan Bahan di Rumah":
        st.header("Masak dengan Bahan di Rumah")
        ingredients_text = st.text_area("Masukkan bahan (pisahkan dengan koma)", placeholder="contoh: telur, tomat, sosis, nasi sisa", height=120)
        time_limit = st.selectbox("Waktu memasak yang diinginkan", ["Any", "< 15 menit", "15-30 menit", "30-60 menit", "> 60 menit"])
        servings = st.number_input("Porsi", min_value=1, max_value=20, value=2)
        diet = st.multiselect("Preferensi diet / pantangan", ["Vegetarian", "Vegan", "No pork", "No garlic/onion", "Low carb", "High protein", "No dairy"])

        if st.button("Buat resep"):
            if not api_key_input:
                st.error("Masukkan OpenAI API Key di sidebar atau set environment variable OPENAI_API_KEY.")
            elif not ingredients_text.strip():
                st.error("Masukkan minimal 1 bahan.")
            else:
                with st.spinner("Menghubungi YummyBot..."):
                    prompt_user = (
                        f"Ingredients: {ingredients_text}\n"
                        f"Time preference: {time_limit}\n"
                        f"Servings: {servings}\n"
                        f"Diet: {', '.join(diet) if diet else 'No specific diet'}\n\n"
                        "Tolong berikan: 3 ide menu singkat (judul), lalu untuk menu pilihan ke-1 berikan resep lengkap termasuk bahan dan langkah, estimasi waktu, tingkat kesulitan, tips penyimpanan, dan 2 substitusi bahan jika ada."
                    )
                    messages = [
                        {"role": "system", "content": PROMPT_SYSTEM},
                        {"role": "user", "content": prompt_user},
                    ]
                    try:
                        output = call_openai_chat(api_key_input, messages, model=model, temperature=temperature)
                    except Exception as e:
                        st.error(f"Gagal memanggil API: {e}")
                        output = None

                    if output:
                        st.session_state.history.insert(0, {"mode": mode, "input": ingredients_text, "output": output})
                        st.markdown("**Hasil YummyBot:**")
                        st.write(output)


    elif mode == "Pengganti Bahan":
        st.header("Pengganti Bahan")
        missing = st.text_input("Bahan yang ingin diganti (mis. garam, telur)")
        have = st.text_area("Bahan yang tersedia (opsional)", placeholder="mis. susu, tepung, minyak")
        if st.button("Cari pengganti"):
            if not api_key_input:
                st.error("Masukkan OpenAI API Key di sidebar atau set environment variable OPENAI_API_KEY.")
            elif not missing.strip():
                st.error("Tolong isi nama bahan yang ingin diganti.")
            else:
                with st.spinner("Mencari pengganti..."):
                    prompt_user = (
                        f"Saya butuh pengganti untuk: {missing}.\n"
                        f"Bahan yang saya punya: {have or 'tidak ada informasi'}.\n"
                        "Berikan minimal 3 alternatif pengganti, takaran pengganti relatif (mis. 1 telur = 1/4 cangkir + 1 sdt baking powder), dan catatan kapan pengganti tidak cocok."
                    )
                    messages = [{"role": "system", "content": PROMPT_SYSTEM}, {"role": "user", "content": prompt_user}]
                    try:
                        output = call_openai_chat(api_key_input, messages, model=model, temperature=temperature)
                    except Exception as e:
                        st.error(f"Gagal memanggil API: {e}")
                        output = None

                    if output:
                        st.session_state.history.insert(0, {"mode": mode, "input": missing + ' | ' + have, "output": output})
                        st.markdown("**Saran Pengganti:**")
                        st.write(output)

    elif mode == "Analisis Resep":
        st.header("Analisis Resep")
        recipe_text = st.text_area("Tempel resepmu di sini (bahan + langkah)", height=220)
        analyze_options = st.multiselect("Apa yang ingin dianalisis?", ["Rasa dan keseimbangan", "Penyederhanaan langkah", "Estimasi biaya", "Tips anti-gagal", "Variasi/upgrade"])
        if st.button("Analisiskan resep"):
            if not api_key_input:
                st.error("Masukkan OpenAI API Key di sidebar atau set environment variable OPENAI_API_KEY.")
            elif not recipe_text.strip():
                st.error("Masukkan resep yang ingin dianalisis.")
            else:
                with st.spinner("Menganalisis resep..."):
                    prompt_user = (
                        f"Resep:\n{recipe_text}\n\n"
                        f"Minta: analisis untuk: {', '.join(analyze_options) if analyze_options else 'general analysis'}."
                    )
                    messages = [{"role": "system", "content": PROMPT_SYSTEM}, {"role": "user", "content": prompt_user}]
                    try:
                        output = call_openai_chat(api_key_input, messages, model=model, temperature=temperature)
                    except Exception as e:
                        st.error(f"Gagal memanggil API: {e}")
                        output = None

                    if output:
                        st.session_state.history.insert(0, {"mode": mode, "input": recipe_text, "output": output})
                        st.markdown("**Hasil Analisis:**")
                        st.write(output)

    elif mode == "Planner 7 Hari":
        st.header("Planner 7 Hari")
        family_size = st.number_input("Jumlah orang", min_value=1, max_value=12, value=2)
        budget_label = st.selectbox("Estimasi budget per hari (IDR)", ["<50k","50-100k","100-200k","200-500k", ">500k"])
        cuisine = st.multiselect("Preferensi masakan", ["Indonesia","Western","Asian","Vegetarian","Fusion", "No preference"])
        if st.button("Buat rencana 7 hari"):
            if not api_key_input:
                st.error("Masukkan OpenAI API Key di sidebar atau set environment variable OPENAI_API_KEY.")
            else:
                with st.spinner("Menyusun menu 7 hari..."):
                    prompt_user = (
                        f"Buat rencana makan 7 hari untuk {family_size} orang. Budget: {budget_label}. Preferensi: {', '.join(cuisine) if cuisine else 'No preference'}.\n"
                        "Berikan untuk setiap hari: sarapan, makan siang, makan malam (judul menu), dan daftar belanja ringkas untuk seluruh minggu."
                    )
                    messages = [{"role": "system", "content": PROMPT_SYSTEM}, {"role": "user", "content": prompt_user}]
                    try:
                        output = call_openai_chat(api_key_input, messages, model=model, temperature=temperature)
                    except Exception as e:
                        st.error(f"Gagal memanggil API: {e}")
                        output = None

                    if output:
                        st.session_state.history.insert(0, {"mode": mode, "input": f"family:{family_size}|budget:{budget_label}|cuisine:{cuisine}", "output": output})
                        st.markdown("**Menu 7 Hari:**")
                        st.write(output)

with col2:
    st.subheader("Riwayat & Favorit")
    if st.session_state.history:
        for i, item in enumerate(st.session_state.history[:10]):
            with st.expander(f"{item['mode']} — input: {item['input'][:40]}"):
                st.write(item["output"])
                if st.button("Tambahkan ke Favorit", key=f"fav_{i}"):
                    st.session_state.favorites.append(item)
                    st.success("Disimpan ke favorit!")
    else:
        st.info("Belum ada interaksi. Coba salah satu mode di kiri.")

    st.markdown("---")
    st.subheader("Favorit")
    if st.session_state.favorites:
        for j, fav in enumerate(st.session_state.favorites):
            st.markdown(f"**{j+1}. {fav['mode']} — {fav['input'][:40]}**")
            st.write(fav['output'])
            if st.button("Hapus favorit", key=f"del_{j}"):
                st.session_state.favorites.pop(j)
                st.experimental_rerun()
    else:
        st.info("Belum ada favorit tersimpan.")


# -------------------------
# Optional: Local fallback (simple rule-based) when no API key
# -------------------------
if not api_key_input:
    st.info("Tidak ada API key terdeteksi — YummyBot akan memberi contoh resep statis jika kamu tidak ingin menggunakan API.")
    if st.button("Contoh resep statis: Nasi Goreng Sederhana"):
        st.markdown("**Nasi Goreng Sederhana**")
        st.write("Bahan: nasi sisa, telur, kecap manis, bawang merah, garam, merica, minyak.\nLangkah: panaskan minyak, tumis bawang sampai harum, tambahkan telur orak-arik, masukkan nasi, tambah kecap, garam, merica. Aduk sampai rata. Sajikan.")

# End of file


Overwriting yummy_bot_app.py


In [28]:
import subprocess
import os
import signal
from pyngrok import ngrok, conf
import time


conf.get_default().auth_token = ngrok_token

# Kill previous ngrok tunnels and streamlit
ngrok.kill()

# Kill any previous streamlit running on 8501
!fuser -k 8501/tcp

# Start streamlit
process = subprocess.Popen(["streamlit", "run", "yummy_bot_app.py"])

# Wait for it to spin up
time.sleep(5)

# Start new ngrok tunnel
public_url = ngrok.connect(8501)
print(f"Streamlit app running at: {public_url}")

8501/tcp:            73341
Streamlit app running at: NgrokTunnel: "https://forcibly-grenadierial-nevaeh.ngrok-free.dev" -> "http://localhost:8501"
